## Experimenting with Neo4j graph builder

In [2]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import Neo4jVector
from langchain.embeddings.ollama import OllamaEmbeddings
import os
from langchain.graphs import Neo4jGraph

In [18]:
url = "neo4j+s://0137fd5f.databases.neo4j.io:7687"
username ="neo4j"
password = ""

#the documents are converted to knowledge graphs and added to this server.
graph1 = Neo4jGraph(
    url=url, 
    username=username, 
    password=password
)

In [19]:
graph1.query("DROP INDEX vector IF EXISTS")

[]

In [20]:
from langchain.vectorstores.neo4j_vector import Neo4jVector
from langchain.embeddings import OpenAIEmbeddings
from langchain.graphs import Neo4jGraph
import os

# Set OpenAI API Key
os.environ["OPENAI_API_KEY"] = ""

# Use OpenAI Embeddings
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(model="text-embedding-3-small"),  # or ada-002
    graph=graph1,
    search_type="hybrid",
    node_label="Chunk",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)


In [21]:
response = vector_index.similarity_search(
    "What is community based research"
)
print(response)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL () { ... }} {position: line: 1, column: 1, offset: 0} for query: "CALL { CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score UNION CALL db.index.fulltext.queryNodes($keyword_index, $query, {limit: $k}) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score } WITH node, max(score) AS score ORDER BY score DESC LIMIT $k RETURN reduce(str='', k IN ['text'] | str + '\\n' + k + ': ' + coalesce(node[k], '')) AS text, node {.*, `embedding

[Document(metadata={'position': 8, 'content_offset': 3308, 'page_number': 6, 'fileName': 'CRC-Guidelines-May-12-2021.pdf', 'length': 922}, page_content='\ntext: ”  lives up to the promise. More support is needed  to help this work flourish.  These guidelines, developed by a community- campus collective, offer advice for both  community-based and campus-based people  who want to do collaborative research. This is  an updated and revised version of the 2007  University Neighborhood Partners report,  Guidelines for Community-Based Research.  This new version includes an expanded set of  principles and integrates lessons learned from  the growth of CBR over the last 15 years. WHAT IS COMMUNITY-BASED RESEARCH? The term community-based research refers to  a large family of research approaches, each  with its own history. These approaches were  developed by people working in health, edu\xad cation, activism, social work, community devel\xad opment, human psychology, and many other  areas. Som

In [22]:
keyword_retriever = vector_index.as_retriever(search_type="similarity", search_kwargs={"k": 5})


In [24]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings 

vector_keyword_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-3.5-turbo", temperature=0),  # or "gpt-4"
    chain_type="stuff",
    retriever=keyword_retriever,
    return_source_documents=False
)

In [26]:

vector_keyword_chain.invoke("top 5 keywords for community engaged research")

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL () { ... }} {position: line: 1, column: 1, offset: 0} for query: "CALL { CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score UNION CALL db.index.fulltext.queryNodes($keyword_index, $query, {limit: $k}) YIELD node, score WITH collect({node:node, score:score}) AS nodes, max(score) AS max UNWIND nodes AS n RETURN n.node AS node, (n.score / max) AS score } WITH node, max(score) AS score ORDER BY score DESC LIMIT $k RETURN reduce(str='', k IN ['text'] | str + '\\n' + k + ': ' + coalesce(node[k], '')) AS text, node {.*, `embedding

{'query': 'top 5 keywords for community engaged research',
 'result': '1. Community-based pedagogy\n2. Critical service learning\n3. Collaboration\n4. Reflections\n5. Community input'}

In [27]:
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
graph2.query(
    "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

def generate_full_text_query(input: str) -> str:
    """
    Generate a full-text search query for a given input string.

    This function constructs a query string suitable for a full-text
    search. It processes the input string by splitting it into words and 
    appending a similarity threshold (~2 changed characters) to each
    word, then combines them using the AND operator. Useful for mapping
    entities from user questions to database values, and allows for some 
    misspelings.
    """
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

In [28]:
# Fulltext index query
def structured_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned
    in the question
    """
    result = ""
    entities = entity_chain.invoke({"question": question})
    print(entities.names)
    for entity in entities.names:
        response = graph2.query(
            """CALL db.index.fulltext.queryNodes('entities', $query, 
            {limit:2})
            YIELD node,score
            CALL {
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS 
              output
              UNION
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS 
              output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        # for el in response:
        #     print(el['output'])
        result += "\n".join([el['output'] for el in response if el['output'] is not None])
    return result

In [29]:
print(structured_retriever("Community engaged research work examples"))


NameError: name 'entity_chain' is not defined